# Lab 4: Neural Signal Processing & Decoding II

For this week's lab, we will continue to use the same dataset from last week, but focus on analyzing the Local Field Potential (LFP) data. For detailed introduction of the dataset, please refer to lab3. We will first go through a few methods to visualize the frequency-domain characteristics of the LFP data and assess the different pros and cons. Using the observations from the spectrograms, we will subsequently extract the frequency-domain features and apply them in a binary classification task to decode movement onsets.

<p align="center">
  <img src="assets/LFP.png" alt="Image" width="600">
</p>

## 1. Load and inspect LFP data

Local-field potentials (LFPs): Local field potentials from each electrode. For the purposes of this analysis, we will only analyze LFPs from a subset of the electrodes.

LFP data files (*_lfp.mat) include the following variables:

- LFP – [#electrodes x time] matrix with local-field potentials (voltage, in mV), sampled at 1 kHz
    - Row 1 = LFP time-series for electrode 1
    - Row 2 = LFP time-series for electrode 2
    - …
- EVENTS – [# events x 1] vector of integer “event codes” to denote the identity of each event (e.g. trial start vs. reward delivered). A full list of event codes and their meanings is provided below.
- EVENT_TIMES – [# events x 1] vector of time-stamps for each task event (in seconds)
- FS_lfp – double with the value of the LFP sampling rate

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.io import loadmat

lab_dir = Path.cwd() 
data_dir = lab_dir/"data"
file_base = "jeev050712a_bioe466_winter2020"
RANDOM_SEED = 42


In [ ]:
# Load LFP and Event Data
lfp_data = loadmat(data_dir/f"{file_base}_lfp.mat", squeeze_me=True)
lfp = np.asarray(lfp_data["LFP"])
fs_lfp = float(lfp_data["FS_lfp"])
events = np.asarray(lfp_data["EVENTS"], dtype=int).ravel()
event_times = np.asarray(lfp_data["EVENT_TIMES"], dtype=float).ravel()

print("LFP shape (electrodes, samples):", lfp.shape)
print("Sampling rate (Hz):", fs_lfp)
print("Duration (s):", lfp.shape[1]/fs_lfp)

In [ ]:
# Reset locally so rerunning this cell selects the same idle timestamps.
rng = np.random.default_rng(RANDOM_SEED)
onsets = np.unique(event_times[(events == 6) & np.isfinite(event_times)])
idles = np.asarray(sorted(rng.choice(np.arange(lfp.shape[1]), size=len(onsets)+100, replace=False) / fs_lfp)) 
print(len(idles), len(onsets))

## 2. PSD and 1/f Trend

The power spectral density (PSD) describes how the power of a signal is distributed across different frequencies. Welch’s method provides a robust way to estimate the PSD by dividing the signal into shorter, overlapping segments, computing the spectrum of each segment, and averaging them together. This averaging produces a smoother and more stable estimate of the frequency content of the LFP.

<p align="center">
  <img src="assets/Welch.png" alt="Image" width="500">
</p>

Use Welch’s method to estimate and visualize the power spectral density (PSD) for each electrode. Plot the PSD in two ways: (1) with frequency on a linear scale and power on a logarithmic scale, and (2) with both frequency and power on logarithmic scales.

In [ ]:
#TODO: use Welch method to compute PSD of all electrodes, over the entire length of the signal
# with frequency on a linear scale and power on a logarithmic scale
from scipy.signal import welch

f, P = welch(...) # TODO: Student Code: use Welch method to compute PSD of all electrodes, over the entire length of the signal
P = ...

plt.figure(figsize=(7, 5))
def one_over_f(f, exponent=1.5):
    f_ref = 10
    i_ref = np.argmin(np.abs(f - f_ref))
    C = P[i_ref] * (f[i_ref] ** exponent)
    ideal = C / (f ** exponent)
    return ideal
plt.plot(f, one_over_f(f), linestyle='--') # Plot ideal 1/f Trend
... # TODO: Student Code: plot psd
... # TODO: Student Code: set power to log scale
plt.xlabel("Frequency (Hz)")
plt.ylabel("Power spectral density")
plt.xlim(1, 200)
plt.show()

In [ ]:
#TODO: same plot but with both frequency and power on logarithmic scales.

plt.figure(figsize=(7, 5))
plt.plot(f, one_over_f(f), linestyle='--') # Plot ideal 1/f Trend
... # TODO: Student Code: plot psd, with both power and frequency on log scale
plt.xlabel("Frequency (Hz)")
plt.ylabel("Power spectral density")
plt.xlim(1, 200)
plt.show()

**Question 2.1:** Did you observe the 1/f trend in our data? What is the source of this 1/f trend?

**Your Answer Here:**

**Question 2.2:** When does the power deviates from the 1/f trend? How can this information be helpful for the subsequent movement onset decoding?

**Your Answer Here:**

## 3. STFT

Let’s now look at **trial-averaged**, **event-aligned** spectrograms to compare the frequency content of the LFP around movement onset and during idle periods.

1. **leave center** (i.e. marker 6 in last week's lab) - the time at which the animal leaves the center target, which we use here as the movement onset event. The timestamps are recorded in the variables `onsets`, and later used as the positive label in movement onset classification.
2. **idles** - time points at which the animal is not moving. The timestamps are recorded in the variables `idles`, and later used as the negative label in movement onset classification.

For each condition, the corresponding event timestamp is aligned to t=0, so that we can compare how spectral power changes before and after the event.

On top of doing trial-averaging, there are options to do channel-averaging, however time taken will be considerably longer. For speed considerations, the provided code uses a single channel (channel 33), though result should be similar on different channel or channel-averaged versions.

Although these two functions `get_spectrogram` and `plot_trial_averaged_spectrogram` below are relatively long, please take time to read through it and understand every input and output parameters. You will only need to complete a small number of lines in `plot_trial_averaged_spectrogram`, but these functions will be reused throughout the following sections.

In [ ]:
def get_spectrogram(events, lfp, fs_lfp, 
                    t_window, nperseg, noverlap, channel_average,
                    method, **method_kwargs):
    """
    Compute event-aligned spectrograms across trials.

    Returns
    -------
    f : ndarray
        Frequencies bins in Hz.
    t : ndarray
        Time points relative to event onset (t=0).
    trials : ndarray
        Spectrograms across trials, shape (trials, frequency, time).
    """

    # start and end time index of the spectrogram
    t0, t1 = (np.asarray(t_window) * fs_lfp).astype(int)

    # iterate through each trial
    trials = []
    for ts in events:
        idx = round(ts * fs_lfp)
        # Skip incomplete windows at the recording boundaries.
        if idx + t0 < 0 or idx + t1 > lfp.shape[-1]:
            continue

        # if channel_average, compute for spectrogram within the window for all channel and trial
        # if channel_average is int, only compute spectrogram of all trials from a specific channel
        if channel_average is True:
            x = lfp[:, idx+t0 : idx+t1]
        else:
            x = lfp[channel_average:channel_average + 1, idx + t0:idx + t1]

        # use the spectrogram method as specified. Note that method_kwargs is passed through
        f, t, P = method(x, fs_lfp, nperseg, noverlap, **method_kwargs)

        # handle channel averaging or not
        if channel_average is True:
            P = P.mean(axis=0)
        else:
            P = P[0]
        trials.append(P)

    # make sure centered time is t=0
    t = t + t_window[0]

    return f, t, np.asarray(trials)

In [ ]:
# TODO: complete plot_trial_averaged_spectrogram function

def plot_trial_averaged_spectrogram(
    lfp,
    fs_lfp,
    con1,
    con2,
    t_window=(-2.0, 2.0),
    f_range=(1, 120),
    nperseg=250,
    noverlap=200,
    channel_average=33,
    zscore=True,
    method=None,
    **method_kwargs
):  
    """
    Plot trial-averaged, event-aligned spectrograms for two conditions.

    Parameters
    ----------
    lfp : ndarray
        LFP array of shape (channels, time).
    fs_lfp : float
        Sampling rate in Hz.
    con1, con2 : array-like
        Event timestamps in seconds for the two conditions.
    t_window : tuple
        Time window around each event, in seconds.
    f_range : tuple
        Frequency range to display, in Hz.
    nperseg : int
        Window length in samples passed to `method`.
    noverlap : int
        Number of overlapping samples passed to `method`.
    channel_average : bool or int
        If True, average across channels. If int, use only that channel.
    zscore : bool
        If True, z-score each frequency row across time.
    method : callable
        Spectrogram function returning `(f, t, P)`, where
        `P` has shape (channels, frequency, time).
    **method_kwargs
        Additional arguments passed to `method`.
    """

    # compute trial-averaged (optionally channel-averaged) spectrogram of the two conditions
    f, t, p1 = get_spectrogram(
        con1, lfp, fs_lfp, t_window, nperseg, noverlap, channel_average, method=method, **method_kwargs
    )
    _, _, p2 = get_spectrogram(
        con2, lfp, fs_lfp, t_window, nperseg, noverlap, channel_average, method=method, **method_kwargs
    )

    # only look at specific frequency range
    keep = (f >= f_range[0]) & (f <= f_range[1])
    f = f[keep]
    p1 = p1[:, keep, :].mean(0)
    p2 = p2[:, keep, :].mean(0) 

    # plot two trial-averaged spectrograms, one for each condition
    fig, ax = plt.subplots(1, 2, sharex=True, sharey=True, figsize=(20,5))
    for a, P, title in zip(ax, [p1, p2], ["con1", "con2"]):

        # convert to db
        P = 10 * np.log10(P)

        # optionally z-score each row
        if zscore:
            P = ... # TODO: Student Code: Apply row-wise zscoring if desired

        # finish plotting
        mappable = a.pcolormesh(...) # TODO: Student Code: use pcolormesh to plot the spectrogram
        a.axvline(0, color="k", ls="--")
        a.set(title=title, xlabel="Time (s)")
    ax[0].set_ylabel("Frequency (Hz)")
    fig.colorbar(
        mappable,
        ax=ax,
        label="Z-scored power" if zscore else "Power (dB)"
    )

In [ ]:
#TODO: use stft to calculate the plot the spectrogram
from scipy.signal import stft

def compute_stft(x, fs_lfp, nperseg, noverlap):
    f, t, Z = stft(...) # TODO: Student Code: use scipy.signal.stft to compute stft
    P = np.abs(Z) ** 2  # channel x frequency x time 
    return f, t, P

plot_trial_averaged_spectrogram(lfp, fs_lfp, onsets, idles, channel_average=33, method=compute_stft,)

**Question 3.1:** Compare the result of whether doing `zscore=Ture` or not. Why might me (we) want to look at the spectrogram z-scored row-wise?

**Your Answer Here:**

**Question 3.2:** When computing stft, did you consider boundary/padding? How would the spectrogram look differently without padding?

**Your Answer Here:**

## 4. Multi-Taper 

Multi taper explicitly allows the user to tradeoff spectral resolution for reduced noise estimates. The number of the tapers are controlled by the time-bandwidth product, which determines how much frequency smoothing is introduced. This gives us a principled way to trade frequency resolution for a more stable spectral estimate. Its characteristics are:

- Use many **“tapers”** (windowing functions).

- Tapers are independent, so taking the **PSD** after convolving with separate tapers results in independent PSD estimates.

- More tapers → wider main lobe, greater width of **“nearby frequencies”** to average.

**Procedure:**

1. Decide upon a reasonable **“nearby frequency”** range.  
   Example: `[f-2, f+2]` → `R = 4`

2. Calculate the **time-bandwidth product** (`alpha`):

   $$
   \alpha = \frac{(\text{window length in sec}) \times R}{2}
   $$

3. Number of tapers:

   $$
   \text{Number of tapers} \leq 2\alpha - 1
   $$


In [ ]:
#TODO: use multi-taper to calculate the plot the spectrogram

from scipy.signal.windows import dpss

def compute_multitaper(x, fs_lfp, nperseg, _, NW=3, step=50):
    """
    Extra Parameters
    ----------
    NW : float
        Time-bandwidth product.
    step : int
        Step size between adjacent windows, in samples.
    """

    # generate the tapers
    Kmax = ... # TODO: Student Code: Calculate number of tapersw
    tapers = ... # TODO: Student Code: Use dpss to construct the tapers

    starts = np.arange(0, x.shape[-1] - nperseg + 1, step) 
    f = np.fft.rfftfreq(nperseg, 1 / fs_lfp)
    t = (starts + nperseg / 2) / fs_lfp
    P = []

    # apply generated tapers to each time segment
    for start in starts:
        seg = x[:, start:start + nperseg]
        X = np.fft.rfft(seg[:, None, :] * tapers[None, :, :], axis=-1)
        P.append((np.abs(X) ** 2).mean(axis=1))

    P = np.stack(P, axis=-1) # channel x frequency x time
    return f, t, P

In [ ]:
# TODO: Test different parameter combinations of your choice for mutlitaper method
# In the code comments, show the calculations used to determine each parameter value, e.g.:
# nperseg = 1.0 sec, R=4, time-bandwidthproduct = 2 N_tapers = 3
plot_trial_averaged_spectrogram(lfp, fs_lfp, onsets, idles, channel_average=33, method=compute_multitaper, nperseg=1000, NW=2)

# TODO: Student Code: Test at least three different cominations of parameters
# nperseg = 0.5 sec, R=8, time-bandwidthproduct = 2 N_tapers = 3
plot_trial_averaged_spectrogram(...)
# nperseg = 0.25 sec, R=16, time-bandwidthproduct = 2 N_tapers = 3
plot_trial_averaged_spectrogram(...)
# nperseg = 0.125 sec, R=32, time-bandwidthproduct = 2 N_tapers = 3
plot_trial_averaged_spectrogram(...)


**Question 4.1:** Why does Multitaper allow you to better control the time-frequency tradeoff? How would you control it to obtain better frequency resolution?  How would you control it to obtain better time resolution? 

**Your Answer Here:**

**Question 4.2:** Based on all spectrograms you’ve made aligned to the movement onset (leave center) condition, which features would be best for decoding the transition between rest and movement?

**Your Answer Here:**

**Question 4.3:** Look at the trial-averaged spectrogram on the left for any pattern specific to each frequency band relative to the movement onset. Could you find literatures to justify the pattern you observed? (Hint: what does high-gamma band look like immediately before movement onset? what about beta band after movement onset?)

**Your Answer Here:**

## 5. Wavelet

Morlet wavelets provide another way to examine how the frequency content of a signal changes over time. A Morlet wavelet is constructed by multiplying a complex sinusoid by a Gaussian envelope, creating a short oscillatory waveform localized in both time and frequency. Important computation steps are:

Temporal width of the Gaussian:

$$
\sigma_t = \frac{n_{\mathrm{cycles}}}{2\pi f}
$$

Complex sinusoid:

$$
s(t) = e^{i2\pi ft}
$$

Gaussian envelope:

$$
g(t) = e^{-\frac{t^2}{2\sigma_t^2}}
$$

Morlet wavelet:

$$
\psi(t) = s(t)g(t)
= e^{i2\pi ft}e^{-\frac{t^2}{2\sigma_t^2}}
$$


In [ ]:
#TODO: following the equations above, construct morlet wavelets and visualize them

def construct_morlet(f, fs_lfp, n_cycles=6):

    wavelets = []
    for i, freq in enumerate(f):
        sigma_t = ... # TODO: Student Code: set temporal width of Gaussian envelop
        half = ... # TODO: the Gaussian extends forever, decide where to truncate (-half, half)
        tw = ... # TODO: Student Code: construct time axis of the wavelet

        sinusoid = ... # TODO: Student Code: create the complex sinusoid 
        envelop = ... # TODO: Student Code: create the gaussian envelop
        wavelet = ... # TODO: Student Code: multiply gaussian and envelop to obtain the short packet of oscillation
        wavelet /= ... # normalizes the wavelet to unit energy.
        wavelets.append((half, wavelet))

    return wavelets

fig, ax = plt.subplots(figsize=(10,5))
freqs = [10, 20, 30]
wavelets = construct_morlet(freqs, fs_lfp)
for f, (half, wavelet) in zip(freqs, wavelets):
    times = np.arange(half*2+1) - half
    ax.plot(times, wavelet.real, label=f'{f}Hz', alpha=0.8)
ax.set_xlabel('time (ms)')
fig.legend()
plt.show()

In [ ]:
#TODO: use the generated wavelets to compute the spectrogram

from scipy.signal import fftconvolve

def compute_morlet(x, fs_lfp, nperseg, _, n_cycles=6):

    f = np.fft.rfftfreq(nperseg, 1 / fs_lfp)
    f = f[1:]  # drop 0 Hz
     
    t = np.arange(x.shape[-1]) / fs_lfp
    P = np.empty((x.shape[0], len(f), x.shape[-1]))

    wavelets = construct_morlet(f, fs_lfp, n_cycles=n_cycles)

    for i, (half, wavelet) in enumerate(wavelets):
        for ch in range(x.shape[0]):
            xp = np.pad(x[ch], half, mode="reflect")
            z = ... # TODO: Student Code: use fftconvolve to apply the wavelet on the signal
            z = z[half:-half]

            P[ch, i] = np.abs(z) ** 2

    return f, t, P

plot_trial_averaged_spectrogram(
    lfp, fs_lfp, onsets, idles, channel_average=33, method=compute_morlet
)

**Question 5.1:** In what way does the spectrogram computed using wavelets look visually different from the ones computed using multitaper and stft? 

**Your Answer Here:**

**Question 5.2:** Compared to the multitaper method and the stft, does Morlet Wavelet has the same time-frequency tradeoff properties?

**Your Answer Here:**

**Question 5.3:** Compared to the multitaper method and the stft, what properties of the wavelet would make it better for decoding movement onset, and what would make it worse?

**Your Answer Here:**

## 6. Decode Movement Onset

Finally, we are at the decoding section! You’re goal is to build a binary classifier that on every trial detects the onset of a movement state (`onsets`) versus an idle state (`idles`). You are free to choose feature extraction strategy and decoders, with a few instructions:

- **Use less then equal to 10 features.**
- Please only modify the `extract_feature` and `decode` function provided below (they are alreay filled in with a baseline strategy). Use the provided `train_and_evaluate` function to handle everything else (train-test split, cross-validation, etc.).
- Restrict the time window to be within t=-2s to t=2s, centered on the event onsets. However, using a causal window is encouraged (i.e. use only data from t=-2s to t=0s).
- You are not restricted to use any channel, either on one channel or all channels are fine.
- Reusing the previously written functions like `get_spectrogram` is encouraged.

Some ideas to get you started:
- Look at the previous PSD plots. What frequency band contains information we can utilize?
- Look at the previous spectrograms. What windows should we select? If we are using multiple frequency bands, should we use different windows for different frequency bands?

Try to get as high score as possible and aim for an AUROC above 0.80 (cross-validated and averaged on the held-out set). Good luck and have fun!

In [ ]:
#TODO: Modify these two functions to experiment with different feature extraction and decoding methods

import numpy as np
from scipy.signal import welch

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

#TODO: Student Code: Replace with your own implementation
def extract_feature(event_times, lfp, fs_lfp):

    features = []

    for ts in event_times:

        # extract time (0 to 0.5 s)
        start = int(ts * fs_lfp)
        end = int((ts + 0.5) * fs_lfp)

        # single channel
        x = lfp[33, start:end]

        # frequency power
        f, P = welch(
            x,
            fs=fs_lfp,
            nperseg=min(250, x.shape[-1])
        )

        # beta power (13-30 Hz)
        keep = (f >= 13) & (f < 30)
        beta_power = P[keep].mean()

        features.append([beta_power])

    return np.asarray(features)


#TODO: Student Code: Replace with your own implementation
def decode(X_train, y_train, X_test):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    model = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)[:, 1]
    return y_pred, y_score

In [ ]:
from sklearn.model_selection import StratifiedKFold
from scipy.stats import t

def train_and_evaluate(extract_feature, decode):

    X_onset = extract_feature(onsets, lfp, fs_lfp)
    X_idle  = extract_feature(idles, lfp, fs_lfp)
    print(f'n_features: {X_onset.shape[1]}')

    X = np.vstack([X_onset, X_idle])
    y = np.concatenate([
        np.ones(len(X_onset)),
        np.zeros(len(X_idle))
    ])

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_SEED
    )

    aurocs = []
    accuracies = []

    for train_idx, test_idx in cv.split(X, y):

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        y_pred, y_score = decode(
            X_train, y_train, X_test
        )

        accuracies.append(
            accuracy_score(y_test, y_pred)
        )

        aurocs.append(
            roc_auc_score(y_test, y_score)
        )

    aurocs = np.asarray(aurocs)
    accuracies = np.asarray(accuracies)

    mean_auc = aurocs.mean()
    sem_auc = aurocs.std(ddof=1) / np.sqrt(len(aurocs))
    ci_auc = t.ppf(0.975, df=len(aurocs) - 1) * sem_auc

    print(f"Accuracy: {accuracies.mean():.3f}")
    print(f"AUROC: {mean_auc:.3f} +/- {ci_auc:.3f} (95% CI)")

In [ ]:
train_and_evaluate(extract_feature, decode)

**Question 6.1:** Explain your Design Choices and interpret your results.

a) What features did you choose? Did you make any modifications to the way the feature was calculated? What changes improved performance vs. made it worse?

b) What classification strategy did you choose? Did you make any changes to the strategy? How did the changes improve performance vs. make it worse?

c) Generally, what tradeoffs did you encounter while attempting to improve performance?

**Your Answer Here:**

a)

b)

c) 